# Archived-result reproduction
Set `RAFTUP_EXPERIMENT_ROOT` to the revision data archive root before running. This notebook redraws saved results, not model training. Outputs are separate from the archive. See ../README.md.


# Deep-STARmap: eight-method reconstruction comparison

从已有结果重绘；不训练、不重新配准。按顺序运行全部 cell。

输入根目录在第一个代码 cell 的 `ROOT`；所有新结果写入本文件夹的 `outputs/`，不覆盖原始结果。主要绘图代码已复制进本 notebook，可直接修改。

原始文件：
- `/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/alternative_pair_benchmarks/Eight_methods_nRMSE_comparison.ipynb`
- `/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/alternative_pair_benchmarks/method_comparison_helpers.py`

论文对应图片：`eight_methods_combined.png`。

本版分三个绘图 cell：nRMSE 热图、平均值柱状图、3D 重建图。先运行共用数据和函数 cell，再独立重画任一图。所有图使用紧裁剪和零外围 padding，输出到 `outputs/02_3d_reconstruction/panels/`。


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_reviewer_figures_mpl')
# Input project root. Change only this path when moving the project.
ROOT = Path(os.environ.get('RAFTUP_EXPERIMENT_ROOT', './revision_data')).expanduser().resolve()
NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
OUT = RESULTS_DIR / '02_3d_reconstruction' / 'panels'
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Missing input project: {ROOT}'
print('Input project:', ROOT)
print('Outputs:', OUT)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Shared typography and export settings.
FONT_SIZE = 17
TITLE_SIZE = 22
ANNOTATION_SIZE = 16
RECON_TITLE_SIZE = 26
DPI = 300
plt.rcParams.update({
    'font.family': 'Arial', 'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE, 'axes.titlesize': TITLE_SIZE,
    'xtick.labelsize': FONT_SIZE, 'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE, 'pdf.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0,
})

def save_panel(fig, stem):
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(OUT / f'{stem}.{ext}', dpi=DPI,
                    bbox_inches='tight', pad_inches=0, facecolor='white')
    fig.savefig(OUT / f'{stem}_preview.png', dpi=100,
                bbox_inches='tight', pad_inches=0, facecolor='white')
    plt.show()
    plt.close(fig)
    print('Saved:', OUT / f'{stem}.png')


## What exactly is nRMSE here?

The metric is **tissue-normalized root-mean-square XY position error**, named `tissue_nRMSE` in earlier notebooks. Each cell is matched to its ground-truth counterpart by **cell ID within the same slice**. It is not a nearest-neighbor, expression, or embedding-space error.

Let $\hat{\mathbf{x}}_{si}$ be the reconstructed XY position of cell $i$ in slice $s$, and $\mathbf{x}^{*}_{si}$ its known correct position in the **section_00 input reference frame**. The original unrotated ground truth is transformed once by section_00's known rotation/translation to define this common frame. We never fit an extra transformation to make each predicted slice agree with ground truth.

First compute one scale for the entire experiment from the union of **all 15 original, unrotated slices**:

$$L=\sqrt{(x_{\max}-x_{\min})^2+(y_{\max}-y_{\min})^2}.$$

Then, for $N_s$ cells in slice $s$:

$$\operatorname{nRMSE}_s=\frac{1}{L}\sqrt{\frac{1}{N_s}\sum_{i=1}^{N_s}\|\hat{\mathbf{x}}_{si}-\mathbf{x}^{*}_{si}\|_2^2}.$$

This takes the mean **squared Euclidean distance per cell**, then its square root. There is no additional division by two for the two coordinates. The denominator is identical across methods and slices; it does not depend on predicted bounding boxes or independently rotated input boxes.

- **Lower is better; zero means exact reconstruction.** It is dimensionless and is not restricted to [0, 1].
- Tables store fractions. The PNG displays **percent of tissue diagonal**: `100 × nRMSE`. For example, 0.01 is 1%, not an accuracy of 99%.
- Here $L\approx18{,}875.9925$ source-coordinate units, so nRMSE=0.01 corresponds to an RMSE of about 188.76 source units. These units should not be called µm without a verified XY calibration.
- All original cells are evaluated, including unmatched cells and cells removed by PRECAST filtering but later transformed by its slice-level rigid map. Exact coverage and unique cell IDs are validated.
- Z is nominal slice position and is **not evaluated**. This is an XY reconstruction score, not an estimate of 3D depth accuracy.
- Section_00 is the fixed reference. Its value is shown for verification but **excluded from all aggregate comparisons**.


In [ ]:
"""Recompute comparable tissue-normalized XY errors from saved reconstructions."""
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
import anndata as ad

METHODS=['RAFT-UP','PASTE','SPACEL','STAlign','STAligner','GPSA','DeepST','PRECAST']
NAMES=[f'section_{i:02d}' for i in range(15)]

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''):h.update(b)
    return h.hexdigest()

def sources(root):
    root=Path(root);b=root/'all_exp/alternative_pair_benchmarks';d=root/'2_3d_reconstruction_exp/Deep-STARmap'
    return {
      'RAFT-UP':root/'all_exp/deep_starmap/rigid_3d_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'PASTE':d/'paste_deep_starmap_user_merfish_code/rigid_3d_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'SPACEL':b/'SPACEL/all_slices_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'STAlign':b/'STalign/all_slices_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'STAligner':b/'STAligner/all_slices_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'GPSA':d/'all_exp/alternative_pair_benchmarks/GPSA/joint_cuda/all_slices_coordinates.csv.gz',
      'DeepST':b/'DeepST/embedding_rigid/all_slices_reconstruction/all_slices_rigid_coordinates.csv.gz',
      'PRECAST':b/'PRECAST/joint_embedding_rigid/all_slices_reconstruction/all_slices_rigid_coordinates.csv.gz'}

NOTES={
 'RAFT-UP':'Saved 15-slice rigid reconstruction in section_00 frame.',
 'PASTE':'Saved adjacent-pair PASTE-derived rigid transforms composed into section_00.',
 'SPACEL':'Saved Spoint/Splane/Scube reconstruction; reference-frame adjustment from its workflow.',
 'STAlign':'Only two independent pair pilots located (00–01 and 13–14); no complete 15-slice reconstruction. Pair scores excluded.',
 'STAligner':'Adjacent-pair native subgraph training and ICP; recorded 1000 epochs, MNN=100, radius=150; composed reconstruction.',
 'GPSA':'Joint model; saved run: 3000 epochs, 25 inducing points per view and 25 shared, 50 genes, 5 MC samples, lr=0.01.',
 'DeepST':'Pairwise DeepST embeddings plus added MNN/RANSAC rigid registration; not native coordinate registration.',
 'PRECAST':'Joint 15-slice PRECAST embedding plus added MNN/RANSAC rigid registration; filtered cells receive the slice transform.'}

def compare(root,output,overrides=None):
    root,output=Path(root),Path(output);output.mkdir(parents=True,exist_ok=True)
    d=root/'2_3d_reconstruction_exp/Deep-STARmap';rot=d/'processed/synthetic_10um_random_z_rotation_seed42'
    original=d/'processed/synthetic_10um'
    manifest=json.loads((rot/'rotation_ground_truth.json').read_text())
    ref=next(s for s in manifest['sections'] if s['section']=='section_00')
    r=np.asarray(ref['R_row_vector']);t=np.asarray(ref['translation_xy'])
    truth={};lower=np.full(2,np.inf);upper=np.full(2,-np.inf);gt_sources=[]
    for name in NAMES:
        p=original/(name+'.h5ad');a=ad.read_h5ad(p,backed='r')
        try:
            xy=np.asarray(a.obsm['spatial'],float)[:,:2];ids=a.obs_names.astype(str)
            assert ids.is_unique and np.isfinite(xy).all()
            lower=np.minimum(lower,xy.min(0));upper=np.maximum(upper,xy.max(0))
            truth[name]=(ids,xy@r+t)
            gt_sources.append(dict(section=name,path=str(p),sha256=sha(p)))
        finally:a.file.close()
    L=float(np.linalg.norm(upper-lower));assert L>0
    paths=sources(root);paths.update({k:Path(v) for k,v in (overrides or {}).items()})
    rows=[];status=[];means=[]
    for method in METHODS:
        path=paths[method];entry=dict(method=method,path=str(path),note=NOTES[method])
        if not path.is_file():
            entry.update(status='missing',reason='No full 15-slice reconstruction coordinate file found')
            status.append(entry);continue
        try:
            frame=pd.read_csv(path,dtype={'section':str,'cell_id':str})
            assert not frame.duplicated(['section','cell_id']).any(), 'Duplicate cell identifiers'
            assert set(frame.section)==set(NAMES), 'Incomplete or different slice set'
            assert np.isfinite(frame[['x','y']]).all().all(), 'Nonfinite coordinates'
            config=path.parent/'rigid_transforms.json';scale=1.
            if config.is_file():
                c=json.loads(config.read_text());assert c['reference']=='section_00', 'Different reference'
                scale=1. if c.get('xy_to_um') is None else float(c['xy_to_um'])
                entry['transform_sha256']=sha(config)
            else:
                c=json.loads((path.parent/'run_config.json').read_text())
                assert c['reference']=='section_00' and c.get('status')=='completed'
                assert method=='GPSA', 'Coordinate scale not specified'
            assert np.isfinite(scale) and scale>0
            current=[]
            for name in NAMES:
                ids,gt=truth[name];pred=frame[frame.section==name].set_index('cell_id')
                assert len(pred)==len(ids) and set(pred.index)==set(ids), f'Cell coverage mismatch: {name}'
                delta=pred.loc[ids,['x','y']].to_numpy()/scale-gt
                err2=np.sum(delta**2,axis=1)
                value=float(np.sqrt(err2.mean())/L)
                current.append(dict(method=method,section=name,reference=name=='section_00',n_cells=len(ids),
                                    tissue_nRMSE=value,nRMSE_percent=100*value))
            assert current[0]['tissue_nRMSE']<1e-5, 'Reference frame does not match section_00'
            scored=np.array([v['tissue_nRMSE'] for v in current[1:]])
            means.append(dict(method=method,n_scored_slices=14,n_scored_cells=sum(v['n_cells'] for v in current[1:]),
                mean_nRMSE=float(scored.mean()),rms_nRMSE=float(np.sqrt(np.mean(scored**2))),
                mean_percent=float(scored.mean()*100),rms_percent=float(np.sqrt(np.mean(scored**2))*100)))
            rows.extend(current);entry.update(status='available',reason='',sha256=sha(path))
        except (AssertionError,KeyError,ValueError,OSError) as e:
            entry.update(status='invalid',reason=str(e))
        status.append(entry)
    long=pd.DataFrame(rows);summary=pd.DataFrame(means).set_index('method').reindex(METHODS).reset_index()
    table=long.pivot(index='section',columns='method',values='tissue_nRMSE').reindex(index=NAMES,columns=METHODS)
    table.loc['Mean (01–14)']=summary.set_index('method').mean_nRMSE
    table.loc['RMS (01–14)']=summary.set_index('method').rms_nRMSE
    status=pd.DataFrame(status)
    long.to_csv(output/'per_slice_nrmse_long.csv',index=False)
    table.to_csv(output/'per_slice_and_mean_nrmse.csv')
    summary.to_csv(output/'method_summary.csv',index=False);status.to_csv(output/'method_status.csv',index=False)
    definition=dict(L=L,scale='union of all original unrotated XY bounding boxes',xy_min=lower.tolist(),xy_max=upper.tolist(),
        reference='section_00',scored_slices=NAMES[1:],reference_excluded=True,z_evaluated=False,
        mean='arithmetic mean of 14 per-slice nRMSE values',rms='sqrt(mean(per-slice nRMSE squared)); historical overall metric',
        cells='all original cells; exact per-slice cell-ID matching; no post-hoc truth registration',
        gt_sources=gt_sources,rotation_manifest_sha256=sha(rot/'rotation_ground_truth.json'))
    (output/'comparison_definition.json').write_text(json.dumps(definition,indent=2)+'\n')
    exclusions=[dict(method='PASTE2',reason='Recorded failure during select_overlap_fraction: Error in the EMD resolution: try to increase the number of dummy points. No completed full reconstruction.',evidence=str(d/'paste2_deep_starmap_user_merfish_code/pair_run_summary.csv')),
      dict(method='SPIRAL',reason='No completed 15-slice reconstruction found. Full run was stopped after prolonged training (user-reported ~280 minutes). Saved completed result is only section_13–14, 5 epochs, reference section_13; not comparable to a full section_00-anchored reconstruction. This does not establish an inherent inability of SPIRAL.',evidence=str(d/'all_exp/alternative_pair_benchmarks/SPIRAL/pilot_section_13_section_14/run_config.json'))]
    pd.DataFrame(exclusions).to_csv(output/'excluded_methods.csv',index=False)
    return long,summary,table,status,definition,pd.DataFrame(exclusions)


def plot_reconstructions(status, output, dpi=400, root=None, *, figure=None):
    """Export truth, rotated input and eight methods as a sharp 10-panel PNG.

    Ground truth receives only the known section_00 transform, matching the
    evaluation frame. No per-slice fit or display subsampling is applied.
    """
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    frames = {}
    entries = status.set_index('method')
    for method in METHODS:
        if method not in entries.index or entries.loc[method, 'status'] != 'available':
            continue
        path = Path(entries.loc[method, 'path'])
        if not path.is_file() and root is not None and '/raftup_revision_828/' in str(path):
            path = Path(root) / str(path).split('/raftup_revision_828/', 1)[1]
        frame = pd.read_csv(path, dtype={'section': str, 'cell_id': str})
        config = path.parent / 'rigid_transforms.json'
        scale = 1.0
        if config.is_file():
            scale = json.loads(config.read_text()).get('xy_to_um') or 1.0
        frame[['x', 'y']] = frame[['x', 'y']] / float(scale)
        if set(frame.section) != set(NAMES):
            raise ValueError(f'{method}: expected all 15 sections')
        if not np.isfinite(frame[['x', 'y', 'z_nominal_um']].to_numpy()).all():
            raise ValueError(f'{method}: nonfinite coordinates')
        frames[method] = frame
    if not frames:
        raise ValueError('No available full reconstructions to plot')

    if root is None:
        root = next((p for p in Path(output).resolve().parents
                     if (p / '2_3d_reconstruction_exp/Deep-STARmap/processed').is_dir()), None)
    if root is None:
        raise FileNotFoundError('Pass root=ROOT to locate ground truth and simulated input')
    processed = Path(root) / '2_3d_reconstruction_exp/Deep-STARmap/processed'
    rotated = processed / 'synthetic_10um_random_z_rotation_seed42'
    manifest = json.loads((rotated / 'rotation_ground_truth.json').read_text())
    reference = next(s for s in manifest['sections'] if s['section'] == 'section_00')
    rotation = np.asarray(reference['R_row_vector'], dtype=float)
    translation = np.asarray(reference['translation_xy'], dtype=float)
    for title, folder in [('Ground truth', processed / 'synthetic_10um'),
                          ('Simulated input', rotated)]:
        sections = []
        for name in NAMES:
            a = ad.read_h5ad(folder / (name + '.h5ad'), backed='r')
            try:
                xy = np.asarray(a.obsm['spatial'], dtype=float)[:, :2]
                if title == 'Ground truth':
                    xy = xy @ rotation + translation
                sections.append(pd.DataFrame({
                    'section': name, 'cell_id': a.obs_names.astype(str),
                    'x': xy[:, 0], 'y': xy[:, 1],
                    'z_nominal_um': float(np.mean(a.uns['nominal_z_interval_um']))}))
            finally:
                a.file.close()
        frames[title] = pd.concat(sections, ignore_index=True)
    panels = ['Ground truth', 'DeepST', 'SPACEL', 'RAFT-UP', 'PRECAST',
              'Simulated input', 'STAligner', 'GPSA', 'STAlign', 'PASTE']
    coords = pd.concat([f[['x', 'y', 'z_nominal_um']] for f in frames.values()])
    lo, hi = coords.min().to_numpy(), coords.max().to_numpy()
    padding = np.maximum((hi - lo) * .04, 1.0)
    colors = plt.get_cmap('turbo')(np.linspace(.04, .96, len(NAMES)))
    standalone = figure is None
    fig = plt.figure(figsize=(30, 14), facecolor='white') if standalone else figure
    grid = fig.add_gridspec(3, 5, height_ratios=[1, 1, .16]) if not standalone else None
    for i, method in enumerate(panels):
        ax = (fig.add_subplot(2, 5, i + 1, projection='3d') if standalone
              else fig.add_subplot(grid[i // 5, i % 5], projection='3d'))
        ax.set_title(method, fontsize=RECON_TITLE_SIZE, fontweight='bold', pad=6)
        if method not in frames:
            ax.text2D(.5, .5, 'N/A\nNo comparable full reconstruction',
                      transform=ax.transAxes, ha='center')
            ax.set_axis_off()
            continue
        frame = frames[method]
        # One opaque collection lets Matplotlib depth-sort every cell together;
        # translucent per-slice collections made the previous rendering hazy.
        cell_colors = colors[frame.section.map({name: j for j, name in enumerate(NAMES)}).to_numpy()]
        ax.scatter(frame.x, frame.y, frame.z_nominal_um,
                   s=.3, c=cell_colors, alpha=1.0, linewidths=0,
                   edgecolors='none', depthshade=False, rasterized=True)
        ax.set_xlim(lo[0] - padding[0], hi[0] + padding[0])
        ax.set_ylim(lo[1] - padding[1], hi[1] + padding[1])
        ax.set_zlim(lo[2] - padding[2], hi[2] + padding[2])
        ax.set_box_aspect((1, 1, .85))
        ax.view_init(elev=22, azim=-60)
        ax.set_proj_type('ortho')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_zlabel('')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.tick_params(labelsize=10)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.fill = False
            axis._axinfo['grid']['color'] = (.85, .87, .9, .55)
    handles = [Line2D([], [], marker='o', linestyle='', color=color,
                      markersize=8, label=f'{i:02d}') for i, color in enumerate(colors)]
    if standalone:
        fig.legend(handles=handles, title='Section', loc='lower center',
                   bbox_to_anchor=(.5, .015), ncol=15, frameon=False,
                   fontsize=22, title_fontsize=22)
    else:
        legend_ax = fig.add_subplot(grid[2, :])
        legend_ax.set_axis_off()
        legend_ax.legend(handles=handles, title='Section', loc='center',
                         ncol=15, frameon=False, fontsize=22, title_fontsize=22)
    if standalone:
        fig.subplots_adjust(left=0, right=1, top=.96, bottom=.11, wspace=0, hspace=.02)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    path = output / 'eight_methods_3d_reconstructions.png'
    # Export is controlled by the independent 3D plotting cell below.
    return fig, path




In [ ]:
SOURCE_OVERRIDES = {}  # Optional method -> saved coordinate CSV path.
long, summary, table, status, definition, excluded = compare(ROOT, OUT, SOURCE_OVERRIDES)
assert set(status.loc[status.status == 'available', 'method']) == set(METHODS), status
print('Tissue diagonal:', definition['L'])
display(summary)
display(status[['method', 'path', 'status']])


## 1. Per-section nRMSE heatmap


In [ ]:
# Heatmap only. Edit this cell independently after loading the shared data.
import matplotlib as mpl
values = long.pivot(index='method', columns='section', values='nRMSE_percent').reindex(
    index=METHODS, columns=NAMES[1:])
fig, ax = plt.subplots(figsize=(15, 6))
cmap = mpl.colormaps['YlOrRd'].copy()
cmap.set_bad('#e5e7eb')
vmax = max(1., float(np.nanmax(values.to_numpy())))
im = ax.imshow(values.to_numpy(), aspect='auto', cmap=cmap, vmin=0, vmax=vmax)
ax.set_xticks(range(14), [f'{i:02d}' for i in range(1, 15)])
ax.set_yticks(range(len(METHODS)), METHODS)
ax.set_xlabel('Section')
ax.set_title('Per-section nRMSE', loc='left', fontweight='bold', pad=12)
for i in range(len(METHODS)):
    for j in range(14):
        value = values.iloc[i, j]
        ax.text(j, i, 'N/A' if pd.isna(value) else f'{value:.2f}',
                ha='center', va='center', fontsize=ANNOTATION_SIZE,
                color='#5b6470' if pd.isna(value) else ('white' if value > vmax*.6 else '#171717'))
ax.tick_params(labelsize=FONT_SIZE)
cb = fig.colorbar(im, ax=ax, fraction=.035, pad=.025)
cb.set_label('nRMSE (% of tissue diagonal)', fontsize=FONT_SIZE)
cb.ax.tick_params(labelsize=FONT_SIZE)
save_panel(fig, 'nrmse_heatmap')


## 2. Mean nRMSE bar chart


In [ ]:
# Mean across the same 14 non-reference sections, with equal section weights.
vals = summary.set_index('method').reindex(METHODS).mean_percent
colors = ['#1b9e77', '#d95f02', '#7570b3', '#999999',
          '#e7298a', '#66a61e', '#e6ab02', '#1f78b4']
fig, ax = plt.subplots(figsize=(8, 5.7))
positions = np.arange(len(METHODS))
ax.barh(positions, vals.fillna(0), color=colors, height=.7)
ax.set_yticks(positions, METHODS)
ax.invert_yaxis()
ax.set_xlim(0, max(1., float(vals.max())) * 1.30)
for i, value in enumerate(vals):
    ax.text(0 if pd.isna(value) else value, i,
            '  N/A' if pd.isna(value) else f'  {value:.3f}%',
            va='center', fontsize=ANNOTATION_SIZE)
ax.set_xlabel('Mean nRMSE (%)')
ax.set_title('Mean across sections 01–14', loc='left', fontweight='bold', pad=12)
ax.tick_params(labelsize=FONT_SIZE)
ax.spines[['top', 'right']].set_visible(False)
save_panel(fig, 'mean_nrmse_bar')


## 3. Three-dimensional reconstruction comparison

布局、视角和点大小在上方 `plot_reconstructions` 函数中修改。全部方法保持相同坐标范围和视角。


In [ ]:
# Ground truth, simulated input and eight methods; no letter annotations.
fig, _ = plot_reconstructions(status, OUT, root=ROOT, dpi=DPI)
save_panel(fig, '3d_reconstructions')
